# 01 · Wrangling: reading, decoding and repairing the raw data

This project began in 2022 as an attempt to replicate Khomtchouk (2020), who showed that an organism's taxonomic group, and the part of the cell its DNA comes from, can be predicted with surprising accuracy from nothing more than the frequencies of its 64 codons: not their order, not the genes they belong to, just how often each one occurs. I wanted to extend that in one direction. Since several codons encode the same amino acid, summing them gives 21 amino-acid frequencies (20 amino acids plus the stop signal), a third as many features; do they carry the same information?

In 2026 I rebuilt the whole analysis, because the original contained errors serious enough that I withdrew its published results (see `ERRATUM.md`). These notebooks keep the questions and the exploration of 2022, and fix what was wrong; where something was wrong, I say what it was.

The data are the UCI Codon Usage dataset (https://archive.ics.uci.edu/dataset/577/codon+usage), committed unchanged as `data/raw/codon_usage.csv`. In this notebook I read the file, work out what its two label columns actually mean, find and repair two malformed rows, and build the amino-acid features. Anything a later notebook depends on lives in `codon_utils.py`, where it is tested (`tests/`); this notebook shows why each piece is needed.

In [1]:
import pandas as pd

import codon_utils

# low_memory=False reads each column in one pass, so pandas infers one dtype
# per column instead of warning about mixed types
codon_usage = pd.read_csv('../data/raw/codon_usage.csv', low_memory=False)

In [2]:
print(codon_usage.head())
print(codon_usage.size)
print(codon_usage.shape)

  Kingdom  DNAtype  SpeciesID  Ncodons  \
0     vrl        0     100217     1995   
1     vrl        0     100220     1474   
2     vrl        0     100755     4862   
3     vrl        0     100880     1915   
4     vrl        0     100887    22831   

                               SpeciesName      UUU      UUC      UUA  \
0  Epizootic haematopoietic necrosis virus  0.01654  0.01203  0.00050   
1                         Bohle iridovirus  0.02714  0.01357  0.00068   
2             Sweet potato leaf curl virus  0.01974   0.0218  0.01357   
3             Northern cereal mosaic virus  0.01775  0.02245  0.01619   
4           Soil-borne cereal mosaic virus  0.02816  0.01371  0.00767   

       UUG      CUU  ...      CGG      AGA      AGG      GAU      GAC  \
0  0.00351  0.01203  ...  0.00451  0.01303  0.03559  0.01003  0.04612   
1  0.00678  0.00407  ...  0.00136  0.01696  0.03596  0.01221  0.04545   
2  0.01543  0.00782  ...  0.00596  0.01974  0.02489  0.03126  0.02036   
3  0.00992  0.01

This is more or less what I expected: 13,028 observations of 69 variables. With 4 distinct nucleotides and three nucleotides per codon there are 4³ = 64 codons, and 64 of the variables are codon frequencies. The remaining five are descriptive: kingdom, DNA type, species ID, number of codons, and species name.

It will be important to discover what is meant by "DNA type", which means finding a data dictionary for this file. It's also worth verifying that the data types are what they ought to be: integers for the species ID, DNA type and number of codons; text for the kingdom and species name; floating-point numbers for every codon frequency.

In [3]:
# every column that is not a float
codon_usage.dtypes[codon_usage.dtypes != 'float64']

Kingdom          str
DNAtype        int64
SpeciesID      int64
Ncodons        int64
SpeciesName      str
UUU              str
UUC              str
dtype: object

`Kingdom` and `SpeciesName` are text, and `DNAtype`, `SpeciesID` and `Ncodons` are integers, as expected. But two codon columns are not numeric: `UUU` and `UUC` were read as text, so at least one value in each column is something other than a number. That deserves a closer look, and gets one below (*Two malformed rows*). First, though, the labels. The unique values of `Kingdom`:

In [4]:
codon_usage['Kingdom'].unique()

<StringArray>
['vrl', 'arc', 'bct', 'phg', 'plm', 'pln', 'inv', 'vrt', 'mam', 'rod', 'pri']
Length: 11, dtype: str

These are not, for the most part, kingdoms. At a glance one might guess that 'vrl' is viruses and 'bct' bacteria, and the UCI page confirms it: `Kingdom` is the three-letter code of the division of the CUTG database (the Codon Usage Tabulated from GenBank) that each sequence set comes from: 'arc' (archaea), 'bct' (bacteria), 'phg' (bacteriophage), 'plm' (plasmid), 'pln' (plant), 'inv' (invertebrate), 'vrt' (vertebrate), 'mam' (mammal), 'rod' (rodent), 'pri' (primate) and 'vrl' (virus).

Some of these divisions are nested inside others: bacteriophages are viruses, primates and rodents are mammals, and mammals are vertebrates. CUTG follows the GenBank divisions, which file each entry in exactly one place, so 'vertebrate' here really means "vertebrate other than a mammal", 'mammal' means "mammal other than a rodent or primate", and 'virus' means "virus other than a bacteriophage". (The GenBank plant division also holds fungi and algae.) These labels are filing categories, then, rather than a flat taxonomy, and that will matter later, when I read the kingdom confusion matrix in `05_results`. I keep the name `kingdom` for continuity, but it should be read as "CUTG division".

`DNAtype` is an integer code for the genome compartment the sequences come from: 0 genomic (nuclear, or the main chromosome of a prokaryote), 1 mitochondrial, 2 chloroplast, 3 cyanelle, 4 plastid, 5 nucleomorph, 6 secondary endosymbiont, 7 chromoplast, 8 leucoplast, 9 "NA", 10 proplastid, 11 apicoplast, 12 kinetoplast. Code 9 needs some care: it must remain a code, and not quietly become a missing value.

It is a little odd that plasmids appear under `Kingdom` rather than `DNAtype`, since a plasmid is a kind of DNA molecule, not a group of organisms.

Most of the rarer DNA types were new to me in 2022, and a bit of reading turned up the following:
1.  A **cyanelle** is an organelle analogous to the chloroplast, found in glaucophytes, a group of freshwater algae.
2.  **Plastids** are a family of organelles found primarily in plants. They include **chloroplasts**, **chromoplasts** and **leucoplasts**, all of which appear as values of this same variable, which is somewhat confusing: to what DNA, exactly, do the records labelled simply 'plastid' refer?
3.  A **nucleomorph** is a vestigial nucleus found between the membrane pairs of some plastids. It is known in only two groups of organisms.
4.  **Secondary endosymbiosis** occurs when a eukaryotic cell engulfs another eukaryotic cell that had itself already taken in a prokaryote; the inner prokaryote is the **secondary endosymbiont**.
5.  **Chromoplasts** are plastids that synthesize and store carotenoid pigments, usually in flowers and fruit.
6.  **Leucoplasts** are plastids used for synthesis and storage, usually of macronutrients; they lack the pigments and the photosynthesis of chloroplasts and chromoplasts, and have a more 'amoeboid' shape.
7.  A **proplastid** is an undifferentiated plastid, formed generally in the meristem tissue of plants; all the specialized plastids derive from them.
8.  **Apicoplasts** are plastids found in most, though not all, apicomplexan parasites. They are probably the product of secondary endosymbiosis, derived from a photosynthetic plastid, and they keep several plant-like properties that make them attractive drug targets. That matters, because the organisms carrying them include serious pathogens: the parasite that causes malaria, for example.
9.  A **kinetoplast** is a network of circular DNA inside a single large mitochondrion, carrying many copies of the mitochondrial genome. Kinetoplasts are known only in one group of flagellate protists.

With the codes understood, it would be informative to look at how each label is distributed.

In [5]:
codon_usage['Kingdom'].value_counts()

Kingdom
bct    2920
vrl    2832
pln    2523
vrt    2077
inv    1345
mam     572
phg     220
rod     215
pri     180
arc     126
plm      18
Name: count, dtype: int64

It looks like we have a substantial number of cases for each 'Kingdom' other than plasmids. (Plasmids are DNA molecules found in bacteria that are not part of the bacterial chromosome and replicate independently; bacteria can exchange plasmids with each other, in a process called conjugation.) Since plasmid DNA is unlike everything else in this variable, and there are a mere 18 observations of it, it seems appropriate to leave these cases out of the models; `03_preprocessing` does so.

In [6]:
codon_usage['DNAtype'].value_counts()

DNAtype
0     9267
1     2899
2      816
4       31
12       5
3        2
9        2
5        2
11       2
6        1
7        1
Name: count, dtype: int64

Almost all observations fall into categories 0, 1 and 2: genomic, mitochondrial and chloroplast DNA. With only a handful of observations of every other type, and some unanswered questions about how those categories relate to one another (several are subsets of others), it is reasonable to restrict the models to these three.

Of course, another important step is to check how many missing values we have.

In [7]:
codon_usage.isna().values.sum()

np.int64(0)

There are no missing values, as parsed. No missing values, however, is not the same thing as every value being valid, and we already know that `UUU` and `UUC` contain something that isn't a number. Before chasing that down, I'll decode the two label columns with the UCI codebooks. The codebooks live in `codon_utils.py`, so that every notebook uses exactly the same mapping.

In [8]:
codon_usage['kingdom'] = codon_usage['Kingdom'].map(codon_utils.KINGDOM_CODES)
codon_usage['dna_type'] = codon_usage['DNAtype'].map(codon_utils.DNATYPE_CODES)
print('codes missing from the codebooks:', codon_usage[['kingdom', 'dna_type']].isna().sum().sum())
pd.crosstab(codon_usage['kingdom'], codon_usage['dna_type'])

codes missing from the codebooks: 0


dna_type,apicoplast,chloroplast,chromoplast,cyanelle,genomic,kinetoplast,mitochondrial,nucleomorph,plastid,secondary_endosymbiont,unknown
kingdom,,,,,,,,,,,
archaea,0,0,0,0,126,0,0,0,0,0,0
bacteria,0,0,0,0,2918,0,0,0,1,1,0
bacteriophage,0,0,0,0,220,0,0,0,0,0,0
invertebrate,2,1,0,0,922,5,411,1,3,0,0
mammal,0,0,0,0,102,0,470,0,0,0,0
plant,0,815,1,2,1523,0,152,1,27,0,2
plasmid,0,0,0,0,18,0,0,0,0,0,0
primate,0,0,0,0,83,0,97,0,0,0,0
rodent,0,0,0,0,59,0,156,0,0,0,0


The raw codes stay in the frame next to the decoded labels, so nothing is lost. Note that `DNAtype` 9 becomes `'unknown'`, and not the codebook's "NA". This was one of the 2022 bugs: I wrote the string `'NA'` to a CSV file, `pd.read_csv` read it back as a missing value, and the two rows concerned (*x Doritaenopsis* sp. and *x Tritordeum* sp.) were later dropped by `dropna()` with no record that it had happened. They would have been dropped anyway, as a rare class, but the removal should be deliberate and counted, not a side effect of parsing.

The table also shows which rows `03_preprocessing` will remove: the 18 plasmids, the 46 rows of rare DNA types (including those two code-9 rows), and one invertebrate "chloroplast" row (*Toxoplasma gondii*, which turns out to be an apicoplast).

## Two malformed rows

`UUU` and `UUC` were read as text. Which values, exactly, are not numbers?

In [9]:
bad = (pd.to_numeric(codon_usage['UUU'], errors='coerce').isna()
       | pd.to_numeric(codon_usage['UUC'], errors='coerce').isna())
codon_usage.loc[bad, ['SpeciesName', 'Ncodons', 'UUU', 'UUC', 'UUA', 'UAA', 'UAG', 'UGA']]

,SpeciesName,Ncodons,UUU,UUC,UUA,UAA,UAG,UGA
486,Non-A,1238,non-B hepatitis virus,0.04362,0.0210,0.02262,0.00162,0.00000
5063,Salmonella enterica subsp. enterica serovar 4,1698,12;I,-,0.0212,0.04476,0.02473,0.00118


There are two such rows, and they share a cause. Both species names contain commas, but neither is quoted in the file: "Non-A,non-B hepatitis virus" (data row 486) and "Salmonella enterica subsp. enterica serovar 4,12;I,-" (row 5063). The CSV reader therefore split each name into two or three fields. Every line of the file still has 69 fields, so the extra pieces of the name pushed the codon values one or two columns to the right, and the last one or two codon values fell off the end of the line altogether.

In 2022 I found the text in these columns, replaced it with NaN, and then filled the three resulting gaps with zeros, on the grounds that three values out of 13,028 weren't worth more effort. The trouble is that the problem was never three values: it was two entire rows with every codon value in the wrong column, and the zero-fill kept them in the modelling data in exactly that state.

In fact, the missing values can be recovered exactly. Each frequency is a codon count divided by `Ncodons`, rounded to 5 decimals, so multiplying back gives whole-number counts, and the count of a missing codon is just `Ncodons` minus the counts that are present. In row 486, whose name took one extra field, the columns `UUC` to `UGA` hold the values of `UUU` to `UAG`, and the value of `UGA` is missing:

In [10]:
row = codon_usage.loc[486]
present = pd.to_numeric(row[codon_utils.CODONS[1:]])  # the 63 values UUU..UAG
counts = present * row['Ncodons']
print('all counts are whole numbers:', bool((counts - counts.round()).abs().max() < 0.05))
print('codons accounted for:', int(counts.round().sum()), 'of', row['Ncodons'])
print('so UGA =', (row['Ncodons'] - int(counts.round().sum())), '/', row['Ncodons'],
      '=', round((row['Ncodons'] - counts.round().sum()) / row['Ncodons'], 5))

all counts are whole numbers: True
codons accounted for: 1235 of 1238
so UGA = 3 / 1238 = 0.00242


But how do I know that it's the *last* codons that are missing, and not the first? Suppose instead that the values were aligned to the right. Then row 486 would use the stop codon UAA for 2.3% of all its codons, and row 5063 would use UAA for 4.5% and UAG for 2.5%. Stop codons occur once per gene, at the end, so they are typically a few tenths of a percent of all codons; aligned to the left, the two rows have 0.40% and 0.12% stop codons, which is entirely ordinary. Left alignment is the only reading that makes biological sense.

`codon_utils.read_raw` applies this repair (`repair_row`) to any row whose codon fields are not all numeric, and raises an error for any row it can't repair exactly. It also adds the decoded `kingdom` and `dna_type` columns, and a `repaired` flag so that the two rows can always be found again:

In [11]:
raw = codon_utils.read_raw('../data/raw/codon_usage.csv')
raw.loc[raw['repaired'], ['SpeciesName', 'Ncodons', 'UUU', 'UUC', 'UAA', 'UAG', 'UGA']]

,SpeciesName,Ncodons,UUU,UUC,UAA,UAG,UGA
486,"Non-A,non-B hepatitis virus",1238,0.04362,0.02100,0.00162,0.0,0.00242
5063,"Salmonella enterica subsp. enterica serovar 4,...",1698,0.02120,0.02356,0.00118,0.0,0.00000


In [12]:
sums = raw[codon_utils.CODONS].sum(axis=1)
print(f'codon frequencies sum to between {sums.min():.5f} and {sums.max():.5f} in every row')
print(f'row 486 sums to {sums[486]:.5f}')

codon frequencies sum to between 0.99984 and 1.00017 in every row
row 486 sums to 1.00000


After the repair, every row's codon frequencies sum to 1 within rounding. With the 2022 zero-fill, row 486 summed to 0.99758, an outlier that a row-sum check on the codon columns would have caught; the original check was applied only to the amino-acid columns, where the discrepancy passed for rounding.

## Amino-acid features

Each amino acid is encoded by between one and six synonymous codons, and three codons are stop signals. Summing the codon frequencies within each group gives 21 features: the 20 amino acids and *stop*. The 2022 version also had a 22nd, *start*, which was redundant. AUG is the usual start codon, but it is also the codon for methionine, wherever it appears in a gene; "start" is a role that AUG plays at one position, not a separate codon. This dataset gives one aggregate frequency per codon for each organism, which cannot distinguish an AUG that starts a gene from an AUG in the middle of one, so the *start* column was an exact copy of *methionine*. It has been removed. Because the 21 groups partition the 64 codons, each row's amino-acid frequencies now sum to exactly the same total as its codon frequencies, which the next cell checks.

In [13]:
print(', '.join(f'{amino} ({len(codons)})' for amino, codons in codon_utils.AMINO_CODONS.items()))
amino = codon_utils.amino_features(raw)
amino.head()

alanine (4), arginine (6), asparagine (2), aspartic_acid (2), cysteine (2), glutamine (2), glutamic_acid (2), glycine (4), histidine (2), isoleucine (3), leucine (6), lysine (2), methionine (1), phenylalanine (2), proline (4), serine (6), threonine (4), tryptophan (1), tyrosine (2), valine (4), stop (3)


,alanine,arginine,asparagine,aspartic_acid,cysteine,glutamine,glutamic_acid,glycine,histidine,isoleucine,...,lysine,methionine,phenylalanine,proline,serine,threonine,tryptophan,tyrosine,valine,stop
0,0.08673,0.06817,0.03008,0.05615,0.01404,0.03409,0.05564,0.08722,0.02857,0.03308,...,0.04913,0.02506,0.02857,0.07268,0.06115,0.04561,0.01003,0.02757,0.09423,0.00301
1,0.08548,0.06378,0.03799,0.05766,0.01899,0.02781,0.05970,0.06716,0.02171,0.04545,...,0.03935,0.03324,0.04071,0.05020,0.06581,0.05768,0.01425,0.03392,0.08955,0.00339
2,0.05594,0.07445,0.04751,0.05162,0.02859,0.03887,0.04710,0.05779,0.03291,0.05203,...,0.06026,0.02180,0.04154,0.06232,0.08289,0.05491,0.01728,0.03887,0.05780,0.00535
3,0.05587,0.04596,0.03656,0.05692,0.01880,0.02820,0.06475,0.06423,0.01462,0.07154,...,0.07415,0.02924,0.04020,0.03760,0.09191,0.06215,0.01201,0.03029,0.07885,0.00418
4,0.06601,0.05789,0.04534,0.06631,0.01787,0.03311,0.07038,0.05658,0.02032,0.04893,...,0.07372,0.02773,0.04187,0.02798,0.06893,0.06745,0.01205,0.03456,0.07196,0.00175


In [14]:
print('columns:', amino.shape[1])
print('largest |amino sum - codon sum|:', (amino.sum(axis=1) - sums).abs().max())
print('methionine is AUG:', bool((amino['methionine'] == raw['AUG']).all()))

columns: 21
largest |amino sum - codon sum|: 1.1102230246251565e-15
methionine is AUG: True


Nothing is written to disk. The later notebooks call `codon_utils.read_raw` themselves, so there is a single source of truth and no derived file to keep in sync with it. (The 2022 version saved an intermediate copy, `codon_usage2.csv`, for the next notebook to read.)